# From one stochastic trajectory to a distribution

Write the equations, choose the numerical method, execute individual samples and compute their statistics in this notebook. It imports no example module and reads no repository data files.

Use a Python kernel with **this development checkout installed** (`python -m pip install .` from its repository root). The released 0.1.3 wheel predates these APIs. All computations below use the CPU. NumPy is installed with Eqiora; Matplotlib is optional for the figure.

We study two dimensionless Ito processes:

- Mean reversion: $dX=2(1/2-X)\,dt+(1/4)\,dW$, $X_0=1$.
- Multiplicative growth: $dY=(1/4)Y\,dt+(1/2)Y\,dB$, $Y_0=1$.

$W$ and $B$ are declared independent Wiener channels. Their increments have variance $\Delta t$, so volatility has units of $s^{-1/2}$. A random increment proportional to $\Delta t$ would describe the wrong process.


In [ ]:
import math
import numpy as np
import eqiora as q

print("Eqiora", q.__version__)
assert hasattr(q, "MomentEstimator"), "Install this development checkout in the notebook kernel."


## 1. Author the laws and initial conditions

`calculus ito` is part of the Model. The solver does not silently reinterpret a Stratonovich law. Each state has an explicit initial value and one evolution law.

The third state, `brownian`, follows the same noise as `growth`. It makes the realized driver visible for the later exact-path comparison. It is not a separate random draw.


In [ ]:
source = r"""
model Processes() {
  noise W: wiener(covariance=1, independent=true);
  noise B: wiener(covariance=1, independent=true);
  state reversion: 1;
  state growth: 1;
  state brownian: s^(1/2);
  initial { reversion=1; growth=1; brownian=0[s^(1/2)]; }
  law restoring for reversion {
    calculus ito;
    drift 2[1/s] * (0.5-reversion);
    diffusion W = 0.25[1/s^(1/2)];
  }
  law growing for growth {
    calculus ito;
    drift 0.25[1/s] * growth;
    diffusion B = 0.5[1/s^(1/2)] * growth;
  }
  law driving for brownian {
    calculus ito;
    drift 0[1/s^(1/2)];
    diffusion B = 1;
  }
  observable reverting_value: 1 = reversion;
  observable growing_value: 1 = growth;
}
"""
model = q.compile(source=source)
reversion = model.field("reversion")
growth = model.field("growth")
brownian = model.field("brownian")
reverting_value = model.observable("reverting_value")
growing_value = model.observable("growing_value")


## 2. Resolve a method and run one sample

At depth 5, Euler–Maruyama uses $2^5=32$ equal steps over the Run horizon. The model, method, output schedule and sampling identity have different roles. The initial State is immutable and can be supplied to independent Runs.

The master seed and full sample/channel identity select a retained path. Reusing that identity replays the same path; changing a sample ID requests another realization. Worker positions and output-table row numbers are not sample IDs.


In [ ]:
depth = 5
plan = q.resolve(model, temporal=q.time.EulerMaruyama(depth=depth))
initial = q.State.initial(plan)
master_seed = bytes([83] * 32)
output_times = tuple(index / 16 for index in range(1, 17))
identity = q.SamplingIdentity(b"one-path", b"lesson")
path = q.run(
    plan, state=initial, until_s=1., output_times_s=output_times,
    master_seed=master_seed, sampling=identity,
)
replayed = q.run(
    plan, state=initial, until_s=1., output_times_s=output_times,
    master_seed=master_seed, sampling=identity,
)
for field in (reversion, growth, brownian):
    np.testing.assert_array_equal(path.series(field).values, replayed.series(field).values)
print("One endpoint:", float(path.series(reversion).values[-1]), float(path.series(growth).values[-1]))


## 3. Derive the distribution before sampling it

For mean reversion with parameters $\lambda,\mu,\sigma$, the continuous mean and variance are

$$m_X(t)=\mu+(X_0-\mu)e^{-\lambda t},\qquad v_X(t)=\frac{\sigma^2}{2\lambda}(1-e^{-2\lambda t}).$$

For geometric Brownian motion, Ito's formula gives

$$Y_t=Y_0\exp((a-b^2/2)t+bB_t),$$

hence $m_Y(t)=Y_0e^{at}$ and $v_Y(t)=Y_0^2 e^{2at}(e^{b^2t}-1)$.

An individual path fluctuates around these distributions. The mean is not the prediction for every realization.


In [ ]:
def exact_moments(t):
    ou_mean = .5 + .5 * math.exp(-2*t)
    ou_variance = .25**2 * (1-math.exp(-4*t)) / 4
    gbm_mean = math.exp(.25*t)
    gbm_variance = math.exp(.5*t) * (math.exp(.25*t)-1)
    return ou_mean, ou_variance, gbm_mean, gbm_variance

print("Continuous endpoint mean/variance:", exact_moments(1.))
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print("Install matplotlib to draw the optional figure.")
else:
    figure, axes = plt.subplots(1, 2, figsize=(10, 3))
    for axis, field, column, title in (
        (axes[0], reversion, 0, "Mean reversion"),
        (axes[1], growth, 2, "Multiplicative growth"),
    ):
        axis.plot(output_times, np.asarray(path.series(field).values), label="one sample")
        axis.plot(output_times, [exact_moments(t)[column] for t in output_times], label="continuous mean")
        axis.set(xlabel="time (s)", ylabel="dimensionless value", title=title)
        axis.legend()
    figure.tight_layout()


## 4. Execute a declared finite ensemble

The loop below runs the ordinary lifecycle once per member. Each Run has its own accepted physical and stochastic state. A failed Run raises an error; the loop does not remove it and recompute a statistic on the survivors.

128 samples make a small lesson, not the scientific acceptance campaign. The registered evidence uses 4096 identities and fixed analytical criteria. Independence and finite moments are explicit statistical assumptions; different identity bytes alone cannot prove them.


In [ ]:
sample_count = 128
samples = [q.SamplingIdentity(index.to_bytes(4, "little"), b"ensemble-lesson") for index in range(sample_count)]
members = []
for sample in samples:
    members.append(q.run(
        plan, state=initial, until_s=1., output_times_s=(1.,),
        master_seed=master_seed, sampling=sample,
    ))

statistics = {}
for name, output in (("OU", reverting_value), ("GBM", growing_value)):
    mean = q.MomentEstimator.from_time_results(members, [output], time_s=1., assume_iid=True)
    variance = q.MomentEstimator.from_time_results(
        members, [output], time_s=1., moment="variance", assume_iid=True,
    )
    assert mean.sampling_identities == samples
    assert len(mean.result_identities) == sample_count
    statistics[name] = (mean, variance)
    print(f"{name}: mean={mean.value:.6f}, sample variance={variance.value:.6f}, estimated mean SE={mean.standard_error():.6f}")


## 5. Separate sampling uncertainty from the time step

The standard error estimates uncertainty of an IID sample **mean**. It is not a confidence interval and does not estimate time-discretization bias. Increasing the number of samples reduces sampling uncertainty; refining the time grid changes the discretization bias.

For $h=1/n$ the OU Euler recurrence has mean $\mu+(X_0-\mu)(1-\lambda h)^n$ and variance $\sigma^2h\sum_{j=0}^{n-1}(1-\lambda h)^{2j}$. The GBM Euler product has first moment $(1+ah)^n$ and second moment $((1+ah)^2+b^2h)^n$.


In [ ]:
steps = 2**depth
h = 1 / steps
r = 1 - 2*h
finite_grid = {
    "OU": (.5 + .5*r**steps, .25**2*h*(1-r**(2*steps))/(1-r*r)),
    "GBM": ((1+.25*h)**steps, ((1+.25*h)**2+.5**2*h)**steps-(1+.25*h)**(2*steps)),
}
continuum = exact_moments(1.)
for index, name in enumerate(("OU", "GBM")):
    grid_mean, grid_variance = finite_grid[name]
    exact_mean, exact_variance = continuum[2*index:2*index+2]
    print(name, "finite-grid mean/variance:", grid_mean, grid_variance)
    print("  mean discretization bias:", grid_mean-exact_mean)
    print("  variance discretization bias:", grid_variance-exact_variance)
    print("  observed mean minus finite-grid mean:", statistics[name][0].value-grid_mean)


## 6. Compare paths using the same realized noise

Refine the method while retaining the same Model and full sampling identity. The Brownian values at the shared output times should agree up to accumulation roundoff. Compare the GBM endpoint with its exact path formula, including the $-b^2/2$ Ito correction.

A single sample's error need not decrease at every refinement. Strong accuracy is assessed over a fixed collection of coupled paths; the independent registered case checks mean squared error against analytical mixed moments at all three grids.


In [ ]:
coupled_sample = q.SamplingIdentity(b"paired-path", b"refinement-lesson")
coarse_driver = None
path_errors = []
shared_times = tuple(index / 8 for index in range(1, 9))
for refinement in (3, 5, 7):
    refined_plan = q.resolve(model, temporal=q.time.EulerMaruyama(depth=refinement))
    result = q.run(
        refined_plan, state=q.State.initial(refined_plan), until_s=1., output_times_s=shared_times,
        master_seed=master_seed, sampling=coupled_sample,
    )
    driver = np.asarray(result.series(brownian).values)
    if coarse_driver is None:
        coarse_driver = driver
    else:
        np.testing.assert_allclose(driver, coarse_driver, rtol=0., atol=4096*np.finfo(float).eps*(1+np.max(np.abs(coarse_driver))))
    exact_path = math.exp(.25 - .5*.5**2 + .5*float(driver[-1]))
    error = abs(float(result.series(growth).values[-1])-exact_path)
    path_errors.append(error)
    print(f"steps={2**refinement}: endpoint={result.series(growth).values[-1]:.8f}, exact same-path endpoint={exact_path:.8f}, absolute error={error:.6g}")


## 7. Check what a repeated sample means

A repeated identity is useful for replay or explicit coupling. It is not a new independent replica. Empirical statistics may retain repeated occurrences, but the IID interpretation must reject them. The original sampling metadata remains attached even after Result serialization.


In [ ]:
try:
    q.MomentEstimator.from_time_results([members[0], members[0]], [reverting_value], time_s=1., assume_iid=True)
except q.ValidationError:
    print("Repeated sample correctly rejected as an IID ensemble.")
else:
    raise AssertionError("A replay was incorrectly admitted as an independent replica")

restored = q.Result.from_bytes(plan, members[0].to_bytes())
np.testing.assert_array_equal(restored.series(reversion).values, members[0].series(reversion).values)
assert len(path_errors) == 3


## Continue the experiment

- Increase the sample count while keeping the time grid fixed. Compare estimated standard error with the unchanged finite-grid bias.
- Refine the grid while keeping sample identities fixed. Inspect same-path errors separately from distribution moments.
- Change `calculus ito` to `calculus stratonovich`. The current method rejects it rather than omitting or inventing a drift correction.
- Increase volatility or use a coarse grid. Euler–Maruyama does not guarantee positive GBM samples.

For reusable application code, `Eqiora.Stochastic.Processes` supplies the same inspectable OU and GBM laws through the ordinary package resolver. The equations above remain visible here so the notebook can be read and executed from scratch.
